# Assignment 14: Image Classification with CNN


In [1]:
import kagglehub
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

/home/brian/Documents/projects/applied-machine-learning/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download the dataset
path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Path to dataset files:", path)

# Define paths
train_dir_labels = os.path.join(path, "seg_train", "seg_train")

# Constants
IMGSIZE = (128, 128)
CLASS_NAMES = sorted([d for d in os.listdir(train_dir_labels) if os.path.isdir(os.path.join(train_dir_labels, d))])

print("IMGSIZE:", IMGSIZE)
print("Class names:", CLASS_NAMES)

Path to dataset files: /home/brian/.cache/kagglehub/datasets/puneet6060/intel-image-classification/versions/2
IMGSIZE: (128, 128)
Class names: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [3]:
def load_split_images(root_path, split_name, img_size, class_names):
    X, y = [], []
    base_dir = os.path.join(root_path, split_name, split_name)
    for label_name in class_names:
        class_dir = os.path.join(base_dir, label_name)
        for f in sorted(os.listdir(class_dir)):
            if not f.lower().endswith('.jpg'):
                continue
            img_path = os.path.join(class_dir, f)
            img = cv2.imread(img_path)  # BGR
            if img is None:
                continue
            img_resized = cv2.resize(img, img_size)
            X.append(img_resized)
            y.append(class_names.index(label_name))
    X = np.stack(X, axis=0)  # shape: (N, H, W, C)
    y = np.array(y, dtype=np.int64)
    return X, y

X_tr, y_tr = load_split_images(path, 'seg_train', IMGSIZE, CLASS_NAMES)
X_ts, y_ts = load_split_images(path, 'seg_test', IMGSIZE, CLASS_NAMES)

print("Train images shape:", X_tr.shape)
print("Test images shape:", X_ts.shape)

Train images shape: (14034, 128, 128, 3)
Test images shape: (3000, 128, 128, 3)


In [4]:
# Scale and Convert to Tensors
X_tr_scaled = X_tr.astype('float32') / 255.0
X_ts_scaled = X_ts.astype('float32') / 255.0

X_train_tensor = torch.from_numpy(X_tr_scaled).permute(0, 3, 1, 2) 
y_train_tensor = torch.from_numpy(y_tr).long()
X_test_tensor = torch.from_numpy(X_ts_scaled).permute(0, 3, 1, 2)
y_test_tensor = torch.from_numpy(y_ts).long()

batch_size = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
def train_model(model, optimizer, train_loader, device, num_epochs=100):
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            _, preds = outputs.max(1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f}")

In [6]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(X_batch)
            _, preds = outputs.max(1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    test_accuracy = correct / total
    print(f"Test accuracy: {test_accuracy:.4f}")

## Part 1 & 2: CNN with Dropout

As the performance standard deviation decreases, there is less variance in the model's performance across different training runs or data splits. This consistency indicates that the model is robust because:
1. It is not overfitting: High variance can be a symptom of overfitting, where the model memorizes specific noise in the training set. A low standard deviation suggests the model has learned generalizable features that work well regardless of the specific data split.
2. It is reliable: A lower standard deviation provides confidence that the model will perform predictably on new, unseen data, rather than fluctuating wildly based on random initialization or specific data samples.

In [7]:
num_classes = len(CLASS_NAMES)

class SimpleCNNWithDropout(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # feature extractor: conv + pool layers
        self.features = nn.Sequential(
            # we need padding 1 to keep the same output size since we are using a 3x3 kernel, 128 x 128
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # (32, 128, 128)
            nn.ReLU(),
            nn.MaxPool2d(2), # now (32, 64, 64)
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (64, 64, 64)
            nn.ReLU(),
            nn.MaxPool2d(2), # now (64, 32, 32)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),# (128, 32, 32)
            nn.ReLU(),
            nn.MaxPool2d(2), # now (128, 16, 16)
        )
        # classifier: flatten + fully connected with dropout
        self.classifier = nn.Sequential(
            nn.Flatten(), # 16 x 16 x 128 = 32768
            nn.Dropout(0.5),
            nn.Linear(128 * 16 * 16, 256), # 23768 -> 256
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_with_dropout = SimpleCNNWithDropout(num_classes).to(device)
optimizer_dropout = torch.optim.Adam(
    model_with_dropout.parameters(),
    lr=3e-4,          # slightly smaller LR for CNN
    weight_decay=1e-4 # modest L2 regularization
)

print(model_with_dropout)

SimpleCNNWithDropout(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=32768, out_features=256, bias=True)
    (3): ReLU()
    (4): Dropout(p=0.5, inplace=False)
    (5): Linear(in_features=256, out_features=6, bias=True)
  )
)


In [8]:
# Training
train_model(model_with_dropout, optimizer_dropout, train_loader, device, num_epochs=100)

Epoch 10/100 - loss: 0.6145 - acc: 0.7763
Epoch 20/100 - loss: 0.4147 - acc: 0.8516
Epoch 30/100 - loss: 0.2812 - acc: 0.9023
Epoch 40/100 - loss: 0.1886 - acc: 0.9352
Epoch 50/100 - loss: 0.1330 - acc: 0.9541
Epoch 60/100 - loss: 0.0921 - acc: 0.9699
Epoch 70/100 - loss: 0.0689 - acc: 0.9772
Epoch 80/100 - loss: 0.0487 - acc: 0.9838
Epoch 90/100 - loss: 0.0461 - acc: 0.9849
Epoch 100/100 - loss: 0.0456 - acc: 0.9850


In [9]:
# Evaluation
evaluate_model(model_with_dropout, test_loader, device)

Test accuracy: 0.8293


## Part 3: Add Batch Normalization and Early Stopping

In [10]:
class CNNWithBatchNorm(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.features = nn.Sequential(
            # Block 1: 3 -> 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Match output channels of Conv2d
            nn.ReLU(),
            nn.MaxPool2d(2), 

            # Block 2: 32 -> 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),  # Match output channels
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), # Match output channels
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        # Classifier (Same as before)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



model_with_batch_norm = CNNWithBatchNorm(num_classes).to(device)
optimizer_batch_norm = torch.optim.Adam(
    model_with_batch_norm.parameters(),
    lr=3e-4,
    weight_decay=1e-4 
)

print(model_with_batch_norm)

CNNWithBatchNorm(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=32768, out_features=

In [11]:
import copy

def train_model_early_stopping(model, optimizer, train_loader, test_loader, device, num_epochs=100, patience=5):
    criterion = nn.CrossEntropyLoss()
    
    best_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        # Training Phase
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            _, preds = outputs.max(1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        
        # Validation Phase (using test_loader for validation here)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        
        val_loss = val_loss / len(test_loader.dataset)

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}")

        # Early Stopping Logic
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0 # Reset counter
        else:
            patience_counter += 1
            print(f"EarlyStopping counter: {patience_counter} out of {patience}")
            if patience_counter >= patience:
                print("Early stopping triggered")
                break
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model

In [12]:
train_model_early_stopping(model_with_batch_norm, optimizer_batch_norm, train_loader, test_loader, device, num_epochs=100, patience=5)

Epoch 1/100 - Train Loss: 1.2451 Acc: 0.5591 | Val Loss: 0.9055
Epoch 2/100 - Train Loss: 0.8252 Acc: 0.6948 | Val Loss: 0.7315
Epoch 3/100 - Train Loss: 0.6908 Acc: 0.7499 | Val Loss: 0.6432
Epoch 4/100 - Train Loss: 0.6131 Acc: 0.7806 | Val Loss: 0.5438
Epoch 5/100 - Train Loss: 0.5694 Acc: 0.7964 | Val Loss: 0.5466
EarlyStopping counter: 1 out of 5
Epoch 6/100 - Train Loss: 0.5231 Acc: 0.8186 | Val Loss: 0.5320
Epoch 7/100 - Train Loss: 0.4887 Acc: 0.8288 | Val Loss: 0.5406
EarlyStopping counter: 1 out of 5
Epoch 8/100 - Train Loss: 0.4572 Acc: 0.8366 | Val Loss: 0.4819
Epoch 9/100 - Train Loss: 0.4372 Acc: 0.8464 | Val Loss: 0.5903
EarlyStopping counter: 1 out of 5
Epoch 10/100 - Train Loss: 0.4068 Acc: 0.8561 | Val Loss: 0.5580
EarlyStopping counter: 2 out of 5
Epoch 11/100 - Train Loss: 0.3867 Acc: 0.8631 | Val Loss: 0.4438
Epoch 12/100 - Train Loss: 0.3647 Acc: 0.8678 | Val Loss: 0.5082
EarlyStopping counter: 1 out of 5
Epoch 13/100 - Train Loss: 0.3488 Acc: 0.8745 | Val Loss: 1

CNNWithBatchNorm(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Dropout(p=0.5, inplace=False)
    (2): Linear(in_features=32768, out_features=

In [13]:
evaluate_model(model_with_batch_norm, test_loader, device)

Test accuracy: 0.8500
